<br><br>

## 🟢 텍스트 분석  

- 데이터를 가지고 직접 분석해보기  
    - 데이터: 로이터 뉴스, IMDB 영화평론 사이트  
    - 1. 토큰화 -> 토큰을 나눈다. 단어로 나눈다. 바꿔서 처리가 가능핟.  
    - 2. 어휘사전 구축 ->  단어한테 고유 숫자를 준다.   
        - I like star  =>  I:0   like:2   star:2  
        - 어휘 사전이 충분히 커야 한다.  
    - 3. 단어 빈도 수 계산   like:10, star:20, red:4 ......  
    - 4. 문서단어 행렬 => 단어 빈도수를 행렬로 표현한다. 0이 엄청 많다. 희소행렬이라고 한다.  

<br><br>

### 🟡 영화 학습  

- aclImdb 데이터셋를 활용하여 실습을 해볼 것입니다.  

#### ⚫️  한눈에 보기  
aclImdb는 “영화 후기 모음집”으로, 인공지능에게 “이 후기가 기분 좋은 글인지, 나쁜 글인지”를 가르칠 때 많이 쓰는 유명한 연습용 데이터 세트입니다.  


#### ⚫️  왜 중요할까요?  
마치 초등학생이 동화책을 읽으며 “이 동화는 행복한 이야기야, 슬픈 이야기야”를 구분해 보는 것처럼, 컴퓨터도 글의 분위기를 알아야 말을 이해하고 똑똑해집니다. aclImdb는 바로 그 연습 문제집 역할을 합니다.  


#### ⚫️  안에 뭐가 들어 있을까요?  
1. 영화 리뷰 글 50,000개  
2. 각 글에는 “긍정(좋아요)” 또는 “부정(싫어요)” 딱지가 붙어 있음  
3. 훈련용·시험용 폴더가 따로 있어서,  
   - 훈련용(train) 25,000개  
   - 시험용(test) 25,000개  
4. 모든 글은 .txt 파일로 저장됨  


#### ⚫️  비유로 쉽게 이해하기  
- 편지 상자 두 개가 있습니다.  
  - 빨간 상자: “이 영화 최고야!” 같은 기분 좋은 편지  
  - 파란 상자: “이 영화 별로야…” 같은 기분 나쁜 편지  
- 컴퓨터에게 두 상자의 편지를 잔뜩 읽혀서,  
  - 새 편지를 보여주면 “이건 빨간 상자 편지구나!” 하고 맞히도록 훈련하는 것,  
  - 그때 쓰는 ‘편지 뭉치’가 바로 aclImdb입니다.  


#### ⚫️  어디에 쓰이나요?  
- 감정 분석(긍·부정 분류) 모델 만들기  
- 자연어 처리(NLP) 연구 논문·대회 예시 데이터  
- 딥러닝 기초 실습용 과제  


#### ⚫️  여러분이 AI 데이터사이언티스트가 되기 위한 첫걸음  
1. 글을 열어 직접 읽어 보세요. “아, 이런 식으로 긍정·부정을 구분하는구나” 감을 잡을 수 있습니다.  
2. 데이터가 폴더로 잘 정리돼 있으므로, 파일 구조를 눈으로 확인하면 데이터 탐색(EDA) 연습이 됩니다.  
3. 이후 파이썬으로 간단히 불러와 모델을 돌려 보면, “데이터 → 모델 → 결과” 흐름을 체험할 수 있습니다.  


#### ⚫️  기억하면 좋은 포인트  
- 이름에서 ‘acl’은 학회(ACL: Association for Computational Linguistics)에 발표됐다는 뜻, ‘Imdb’는 영화 리뷰 사이트 IMDb에서 가져왔다는 뜻입니다.  
- 텍스트만 있으므로, 이미지나 숫자보다 파일 크기가 작아 연습하기 부담이 적습니다.  
- 처음 공부하실 때, 글이 영어라서 막막할 수 있지만 “긍정·부정 딱지”만 잘 활용해도 모델은 충분히 배웁니다.  


#### ⚫️  마무리 한 줄  
aclImdb는 “컴퓨터에게 영화 후기의 기분을 맞히게 만드는 초·중급 레벨의 필수 연습장”이라고 생각하시면 됩니다.  

In [9]:
###############################################################################
# 0. 도구 챙기기
#    - 어떤 일을 하려면 연필, 지우개, 공책이 필요하듯
#      파이썬에서도 라이브러리를 먼저 불러옵니다.
###############################################################################
from sklearn.datasets import load_files           # 우체통에서 편지를 한웅큼 꺼내는 집게
import pandas as pd                               # 편지를 표(엑셀처럼)로 붙여 주는 풀
import numpy as np                                # 숫자 계산용 자와 계산기 (이번 예시는 거의 안 씀)
from sklearn.feature_extraction.text import CountVectorizer  # 단어를 세어 주는 손가락 셈 도구
from sklearn.linear_model import LogisticRegression           # 편지 내용으로 ‘좋아요/싫어요’를 맞히는 선생님

###############################################################################
# 1. 편지 꺼내기
###############################################################################
# aclImdb/train 폴더 안에는
#   pos/  → “좋아요” 편지
#   neg/  → “싫어요” 편지
# 가 폴더별로 담겨 있습니다.
reviews_train = load_files('data/aclImdb/train')
print(reviews_train.keys)   # {'data', 'target', ...} 같은 구성 확인

# reviews_train.data  : 편지(리뷰)가 바이트 문자열 형태로 들어 있음
# reviews_train.target: 각 편지의 답(1=긍정, 0=부정)
text_train = reviews_train.data
y_train    = reviews_train.target

###############################################################################
# 2. 편지를 공책(표)으로 정리하기
###############################################################################
# 첫 번째 열 : 편지 내용
# 두 번째 열 : 답(좋아요/싫어요)
df = pd.DataFrame(text_train, columns=['review'])  # 열 이름을 'review'로 지정
df['target'] = y_train                             # 새 열 추가

# 공책을 CSV 파일로 저장 (utf-8-sig: 엑셀에서 한글 안 깨지게)
df.to_csv('imdb.csv', encoding='utf-8-sig', index=False)

print(df.head())   # 위에서 5줄 미리 보기

###############################################################################
# 3. 편지 내용 청소하기  ― “<br />” 같은 HTML 표식은 스티커 제거하듯 떼어냅니다.
###############################################################################
# text_train 안 자료는 b'...' 처럼 바이트 문자열입니다.
# 바이트 문자열에서도 replace를 쓸 수 있으니 <br /> 를 빈 문자열로 치환합니다.
text_train = [review.replace(b"<br />", b"") for review in text_train]

###############################################################################
# 4. 단어 세기 : CountVectorizer
###############################################################################
# (1) fit : 모든 편지를 훑어보며 “사전”을 만듭니다.
# (2) transform : 각각의 편지를 “단어 개수 표”로 바꿔 줍니다.
vect = CountVectorizer().fit(text_train)
X_train = vect.transform(text_train)

# 사전에 몇 개 단어가 들어갔는지 확인해 봅니다.
feature_names = vect.get_feature_names_out()
print(f"\n특성(단어) 총 개수 : {len(feature_names)}")
print(f"앞에서 20개 단어 : {feature_names[:20]}")

###############################################################################
# 5. 모델 학습 : LogisticRegression
###############################################################################
# 선생님(모델)에게
#   X_train : 단어 개수 표
#   y_train : 답(0/1)
# 을 보여 주며 학습시킵니다.
model = LogisticRegression(max_iter=1000)  # 반복 횟수를 넉넉히 설정
model.fit(X_train, y_train)

# 훈련 데이터에 대해 맞힌 비율(정확도)을 확인합니다.
train_score = model.score(X_train, y_train)
print(f"\n훈련 데이터 정확도 : {train_score:.3f}")

<built-in method keys of Bunch object at 0x111defe30>
                                              review  target
0  b'I caught this film at the Edinburgh Film Fes...       0
1  b"Based upon the recommendation of a friend, m...       0
2  b'This is not an entirely bad movie. The plot ...       0
3  b"I must confess to not having read the origin...       1
4  b"as the title of this post says, the section ...       2

특성(단어) 총 개수 : 43737
앞에서 20개 단어 : ['00' '000' '000000003' '00001' '000s' '001' '007' '0079' '0080' '0083'
 '00am' '00pm' '00s' '01' '02' '03' '04' '05' '06' '06th']

훈련 데이터 정확도 : 1.000


<br>

### 🟡 네이버 영화  

In [2]:
from sklearn.datasets import load_files 
import pandas as pd 
import numpy as np 
from sklearn.feature_extraction.text import CountVectorizer 
import matplotlib.pyplot as plt 
import re #정규식 처리 하는 라이브러리 

#파일을 읽는다 - 구분자가 탭키다 
#keep_default_na - NaN 값을 None으로 바꾼다
df_train = pd.read_csv("data/aclImdb/ratings_train.txt", delimiter="\t", keep_default_na=False)
print( df_train.head() )
text_train, y_train = df_train["document"].values, df_train["label"].values
print( text_train[:3])
print( y_train[:3])

#text_traind을 벡터화하자 

from konlpy.tag import Okt  #현재 가장 많이 사용하는 형태소 분리 알고리즘 
okt = Okt() 
stop_words = ["아", "..", "?", "있는" ]
def okt_tokenizer(text):
    #특수문자를 제거하기 (한글, 숫자, 영어, \s - 공백  )
    # 정규식 패턴을 사용해서 문자 바꿔치기
    text = re.sub(r"[^\uAC00-\uD7A3\s]", "", text)
    temp = okt.morphs(text)
    #제거할거 있으면 제거시켜서 보내기, 불필요한 단어나 한글자는 삭제시키고 나머지만 
    temp = [word for word in temp if word not in stop_words and len(word)>=2]
    return temp 

for i in range(0, 10):
    print( okt_tokenizer(text_train[i] ))

#CountVectorizer의 tokenizer 매개변수에 우리가 토큰나이저 만들어서 주면된다. 
#한글 토큰나이저로 바꿔치기를 한다. 경고는 무시해도 된다. 
vect = CountVectorizer(tokenizer=okt_tokenizer).fit(text_train)

feature_names = vect.get_feature_names_out() 
print("특성의 개수 ", len(feature_names))
print(feature_names[:20])

X_train = vect.transform(text_train)

from sklearn.linear_model import LogisticRegression
model = LogisticRegression(solver='liblinear')
model.fit(X_train, y_train)
print( model.score(X_train, y_train))


         id                                           document  label
0   9976970                                아 더빙.. 진짜 짜증나네요 목소리      0
1   3819312                  흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나      1
2  10265843                                  너무재밓었다그래서보는것을추천한다      1
3   9045019                      교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정      0
4   6483659  사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...      1
['아 더빙.. 진짜 짜증나네요 목소리' '흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나'
 '너무재밓었다그래서보는것을추천한다']
[0 1 1]


['더빙', '진짜', '짜증나네요', '목소리']
['포스터', '보고', '초딩', '영화', '오버', '연기', '조차', '가볍지', '않구나']
['무재', '밓었', '다그', '래서', '보는것을', '추천']
['교도소', '이야기', '구먼', '솔직히', '재미', '없다', '평점', '조정']
['사이', '몬페', '익살스런', '연기', '돋보였던', '영화', '스파이더맨', '에서', '늙어', '보이기만', '했던', '커스틴', '던스트', '너무나도', '이뻐', '보였다']
['걸음', '부터', '초등학교', '학년', '생인', '살용', '영화', '반개', '아까']
['원작', '긴장감', '제대로', '살려내지못', '했다']
['반개', '아깝다', '나온다', '이응경', '길용우', '생활', '인지', '정말', '해도', '그것', '보단', '낫겟다', '납치', '감금', '반복', '반복', '드라마', '가족', '없다', '연기', '하는', '사람']
['액션', '없는데도', '재미', '안되는', '영화']
['왜케', '평점', '낮은건데', '헐리우드', '화려함에만', '너무', '길들여져', '있나']


/opt/anaconda3/envs/aiBootCamp/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


특성의 개수  17281
['가가' '가감' '가게' '가격' '가고' '가고싶으면' '가고싶을' '가관' '가구' '가구야' '가기' '가까운'
 '가까웠을텐데' '가까이' '가깝다' '가끔' '가나' '가나다' '가나다라' '가난']
0.9648316142661861
